# ECE 57000 Final Project - Training Notebook

**Project:** Detection of AI Images from Screenshots  
**Student:** Syeda Maliha Monowara (PUID 0038305425)  
**Runs on:** Google Colab with a T4 GPU (free tier is sufficient)

## What this notebook does

1. Installs dependencies.
2. Clones `meownager/ai-image-screenshot-detector` from GitHub.
3. Downloads CIFAKE from Hugging Face.
4. Runs the four-configuration ablation study from `src/ablation.py`.
5. Calibrates the best model with temperature scaling.
6. Saves `runs/ablation/results.json` and the best checkpoint to Google Drive.

## Before you run

1. In Colab, go to **Runtime > Change runtime type** and pick **T4 GPU**.
2. Optionally mount Drive in the first cell so checkpoints persist.
3. Run the cells top-to-bottom. Expect ~2-4 hours total on a T4.

## 1. Setup

In [ ]:
# Verify GPU is visible.
!nvidia-smi || echo 'No GPU detected - switch runtime to T4 GPU before continuing.'

In [ ]:
# Install dependencies. torch/torchvision are preinstalled on Colab; reinstall only if needed.
!pip install -q 'datasets>=2.19' 'gradio>=4.0' 'huggingface_hub>=0.23' 'tqdm>=4.66' 'pillow>=10.0'

In [ ]:
# Optional: mount Google Drive to persist runs/ and checkpoints across sessions.
from google.colab import drive
drive.mount('/content/drive')

import os
PERSIST_ROOT = '/content/drive/MyDrive/ece57000_ai_screenshot_detector'
os.makedirs(PERSIST_ROOT, exist_ok=True)
print('Persisting outputs to:', PERSIST_ROOT)

## 2. Clone the repo

In [ ]:
%cd /content
!rm -rf ai-image-screenshot-detector
!git clone https://github.com/meownager/ai-image-screenshot-detector.git
%cd ai-image-screenshot-detector
!ls -la

## 3. Download CIFAKE

In [ ]:
# Full CIFAKE is ~120k images. For first-pass experiments, cap per-class samples
# to keep the run under 3 hours on T4. Remove --max-per-class for the full run.
!python scripts/download_cifake.py --out ./data/cifake --max-per-class 20000

## 4. Run the ablation study

In [ ]:
# Runs all 4 configs: none, screenshot, screenshot_jitter, screenshot_jitter_crop.
# Each config: 5 epochs, AdamW, cosine LR, batch 64. ~25-40 min/config on T4.
!python -m src.ablation \
    --data-root ./data/cifake \
    --out-root ./runs/ablation \
    --epochs 5 \
    --batch-size 64 \
    --lr 3e-4 \
    --num-workers 2

## 5. Evaluate on the held-out test set

In [ ]:
!python -m src.eval cifake \
    --data-root ./data/cifake \
    --checkpoint ./runs/ablation/screenshot_jitter_crop/best.pt \
    --out ./runs/ablation/screenshot_jitter_crop/test_report.json

## 6. Temperature calibration

In [ ]:
!python -m src.calibration \
    --data-root ./data/cifake \
    --checkpoint ./runs/ablation/screenshot_jitter_crop/best.pt \
    --out ./runs/ablation/screenshot_jitter_crop/calibration.json

## 7. Persist outputs to Drive

In [ ]:
import shutil, os
src_root = 'runs'
dst_root = os.path.join(PERSIST_ROOT, 'runs')
if os.path.exists(dst_root):
    shutil.rmtree(dst_root)
shutil.copytree(src_root, dst_root)
print('Copied runs/ to', dst_root)
!ls -la {dst_root}/ablation/screenshot_jitter_crop/

## 8. (Optional) Upload best checkpoint to Hugging Face

Requires a token with **write** permission (https://huggingface.co/settings/tokens). The cell below pushes `best.pt` to `meownager/ai-image-screenshot-detector` so the Gradio Space can load it.

In [ ]:
from huggingface_hub import login, upload_file, create_repo
import os

HF_TOKEN = os.environ.get('HF_TOKEN') or input('Paste Hugging Face write token: ').strip()
login(token=HF_TOKEN)

repo_id = 'meownager/ai-image-screenshot-detector'
create_repo(repo_id=repo_id, repo_type='model', exist_ok=True)

for name in ['best.pt', 'calibration.json', 'metrics.json']:
    p = f'runs/ablation/screenshot_jitter_crop/{name}'
    if os.path.exists(p):
        upload_file(path_or_fileobj=p, path_in_repo=name,
                    repo_id=repo_id, repo_type='model')
        print('Uploaded', name)
print('Model repo: https://huggingface.co/' + repo_id)